In [9]:
import pandas as pd
import numpy as np

# 1. Load the Kaggle dataset
# (Ensure you have downloaded train.csv from the competition page)
df = pd.read_csv('train.csv')

# 2. Basic Preprocessing
# Fill missing values for Age with the median
df['Age'] = df['Age'].fillna(df['Age'].median())
# Fill missing values for Fare
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

# Convert categorical 'Sex' to numeric binary (male: 0, female: 1)
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# Select features and the target label
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
X = df[features].values
y = df['Survived'].values

# 3. Feature Scaling (Standardization)
# This centers the data around 0, which helps the sigmoid function and gradient descent
X = (X - X.mean(axis=0)) / X.std(axis=0)

# 4. Train the Custom Model
# We use the LogisticRegressionModel class defined previously
model = LogisticRegressionModel(learning_rate=0.01, n_iterations=300)
model.fit(X, y)

# 5. Evaluate the Model
predictions = model.predict(X)
accuracy = np.mean(predictions == y) * 100

print(f"Training Accuracy: {accuracy:.2f}%")

Training Accuracy: 78.68%


In [3]:
import numpy as np

# 1. Define the Sigmoid Activation Function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

class LogisticRegressionModel:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        n_samples, n_features = X.shape

        # Initialize parameters
        self.weights = np.zeros(n_features)
        self.bias = 0

        # Gradient Descent
        for _ in range(self.n_iterations):
            # Apply the linear equation
            linear_model = np.dot(X, self.weights) + self.bias

            # Pass the result through the sigmoid activation function
            y_predicted = sigmoid(linear_model)

            # Calculate gradients
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)

            # Update parameters
            self.weights -= self.lr * dw
            self.bias -= self.lr * db

    def predict(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        y_predicted = sigmoid(linear_model)

        # Convert probabilities to binary classes (0 or 1)
        y_predicted_cls = [1 if i > 0.5 else 0 for i in y_predicted]
        return np.array(y_predicted_cls)

In [12]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE

# 1. Load and Preprocess
df = pd.read_csv('train.csv')
df['Age'] = df['Age'].fillna(df['Age'].mean())
df['Fare'] = df['Fare'].fillna(df['Fare'].mean())
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
X = df[features].values
y = df['Survived'].values

# 2. Feature Scaling (Strictly required before PCA)
X_scaled = (X - X.mean(axis=0)) / X.std(axis=0)

# 3. PCA: Drop the 2 components with the least variance
# We have 6 original features, so we keep 6 - 2 = 4 components
n_components_to_keep = X_scaled.shape[1] - 2
pca = PCA(n_components=n_components_to_keep)
X_pca = pca.fit_transform(X_scaled)

# 4. SMOTE: Balance the classes
# This generates synthetic samples for the minority class to equalize representation
smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X_scaled, y)

# 5. Train the Custom Logistic Regression Model
# (Assuming the LogisticRegressionModel class from earlier is defined)
model = LogisticRegressionModel(learning_rate=0.1, n_iterations=3000)

# Train on the PCA-reduced, SMOTE-balanced data
model.fit(X_balanced, y_balanced)

# 6. Evaluate
predictions = model.predict(X_balanced)
accuracy = np.mean(predictions == y_balanced) * 100

print(f"Original dataset shape: {X.shape}")
print(f"PCA reduced shape: {X_pca.shape}")
print(f"SMOTE balanced shape: {X_balanced.shape}")
print(f"Training Accuracy: {accuracy:.2f}%")

Original dataset shape: (891, 6)
PCA reduced shape: (891, 4)
SMOTE balanced shape: (1098, 6)
Training Accuracy: 79.05%


In [13]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU
from imblearn.over_sampling import SMOTE

# 1. Pipeline prep (Mean imputation, No PCA)
df = pd.read_csv('train.csv')
df['Age'] = df['Age'].fillna(df['Age'].mean())   # Changed to Mean
df['Fare'] = df['Fare'].fillna(df['Fare'].mean()) # Changed to Mean
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
X = df[features].values
y = df['Survived'].values

X_scaled = (X - X.mean(axis=0)) / X.std(axis=0)

smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X_scaled, y)

# 2. Build the Neural Network
model_nn = Sequential()

# Hidden Layer 1 (needs input_shape to match your 6 features)
model_nn.add(Dense(32, input_shape=(X_balanced.shape[1],)))
model_nn.add(LeakyReLU(alpha=0.01))

# Hidden Layer 2
model_nn.add(Dense(16))
model_nn.add(LeakyReLU(alpha=0.01))

# Hidden Layer 3
model_nn.add(Dense(8))
model_nn.add(LeakyReLU(alpha=0.01))

# Hidden Layer 4
model_nn.add(Dense(4))
model_nn.add(LeakyReLU(alpha=0.01))

# Output Layer (Sigmoid for binary classification)
model_nn.add(Dense(1, activation='sigmoid'))

# 3. Compile and Train
# binary_crossentropy is the standard loss function for sigmoid outputs
model_nn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model, keeping 20% of the data to validate against overfitting
history = model_nn.fit(
    X_balanced,
    y_balanced,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/100


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.6970 - loss: 0.6593 - val_accuracy: 0.7636 - val_loss: 0.5425
Epoch 2/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7859 - loss: 0.6138 - val_accuracy: 0.7182 - val_loss: 0.4788
Epoch 3/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7973 - loss: 0.5594 - val_accuracy: 0.6864 - val_loss: 0.4714
Epoch 4/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8018 - loss: 0.4904 - val_accuracy: 0.6773 - val_loss: 0.5383
Epoch 5/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8109 - loss: 0.4431 - val_accuracy: 0.7000 - val_loss: 0.5462
Epoch 6/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8132 - loss: 0.4320 - val_accuracy: 0.7182 - val_loss: 0.5483
Epoch 7/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8189 - loss: 0.4241 - val_accuracy: 0.7136 - val_loss: 0.5390
Epoch 8/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8292 - loss: 0.4189 - val_accuracy: 0.7136 - val_loss: 0.

In [14]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# 1. Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Validation Split (To calculate F1 Score locally)
train_split, val_split = train_test_split(train_df, test_size=0.2, random_state=42)

# 3. Calculate Statistics ONLY from training data
age_mean = train_split['Age'].mean()
fare_mean = train_split['Fare'].mean()

def preprocess(df, age_val, fare_val):
    df_clean = df.copy()
    df_clean['Age'] = df_clean['Age'].fillna(age_val)
    df_clean['Fare'] = df_clean['Fare'].fillna(fare_val)
    df_clean['Sex'] = df_clean['Sex'].map({'male': 0, 'female': 1})
    features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
    return df_clean[features].values

# 4. Apply Imputation
X_train = preprocess(train_split, age_mean, fare_mean)
y_train = train_split['Survived'].values

X_val = preprocess(val_split, age_mean, fare_mean)
y_val = val_split['Survived'].values

X_test = preprocess(test_df, age_mean, fare_mean)

# 5. Feature Scaling (Fit on Train, Transform all)
scaler_mean = X_train.mean(axis=0)
scaler_std = X_train.std(axis=0)

X_train_scaled = (X_train - scaler_mean) / scaler_std
X_val_scaled = (X_val - scaler_mean) / scaler_std
X_test_scaled = (X_test - scaler_mean) / scaler_std

# 6. Apply SMOTE to Training Data ONLY
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

# ==========================================
# MODEL 1: CUSTOM LOGISTIC REGRESSION
# ==========================================
# (Assuming LogisticRegressionModel class is defined above)
model_lr = LogisticRegressionModel(learning_rate=0.1, n_iterations=3000)
model_lr.fit(X_train_balanced, y_train_balanced)

# Evaluate F1
lr_val_preds = model_lr.predict(X_val_scaled)
lr_f1 = f1_score(y_val, lr_val_preds)
print(f"Logistic Regression Validation F1 Score: {lr_f1:.4f}")

# Predict on test.csv & save
lr_test_preds = model_lr.predict(X_test_scaled)
submission_lr = pd.DataFrame({'PassengerId': test_df['PassengerId'], 'Survived': lr_test_preds})
submission_lr.to_csv('submission_logistic_regression.csv', index=False)


# ==========================================
# MODEL 2: NEURAL NETWORK
# ==========================================
model_nn = Sequential([
    Dense(32, input_shape=(X_train_balanced.shape[1],)),
    LeakyReLU(alpha=0.01),
    Dense(16),
    LeakyReLU(alpha=0.01),
    Dense(8),
    LeakyReLU(alpha=0.01),
    Dense(4),
    LeakyReLU(alpha=0.01),
    Dense(1, activation='sigmoid')
])

model_nn.compile(optimizer='adam', loss='binary_crossentropy')
# Train the model silently (verbose=0)
model_nn.fit(X_train_balanced, y_train_balanced, epochs=100, batch_size=32, verbose=0)

# Evaluate F1
# NN outputs probabilities, so we round them to 0 or 1
nn_val_probs = model_nn.predict(X_val_scaled).flatten()
nn_val_preds = (nn_val_probs > 0.5).astype(int)
nn_f1 = f1_score(y_val, nn_val_preds)
print(f"Neural Network Validation F1 Score: {nn_f1:.4f}")

# Predict on test.csv & save
nn_test_probs = model_nn.predict(X_test_scaled).flatten()
nn_test_preds = (nn_test_probs > 0.5).astype(int)
submission_nn = pd.DataFrame({'PassengerId': test_df['PassengerId'], 'Survived': nn_test_preds})
submission_nn.to_csv('submission_neural_network.csv', index=False)

Logistic Regression Validation F1 Score: 0.7898


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
Neural Network Validation F1 Score: 0.7778
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


In [15]:
# 1. Load the Kaggle test data
test_df = pd.read_csv('test.csv')

# 2. Preprocess using the exact same logic as training
# (Fill missing values with the TRAIN set's mean, map Sex to 0/1)
test_clean = test_df.copy()
test_clean['Age'] = test_clean['Age'].fillna(age_mean)
test_clean['Fare'] = test_clean['Fare'].fillna(fare_mean)
test_clean['Sex'] = test_clean['Sex'].map({'male': 0, 'female': 1})

# Extract features
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
X_test = test_clean[features].values

# 3. Scale using the TRAIN set's mean and std
X_test_scaled = (X_test - scaler_mean) / scaler_std

# ==========================================
# SUBMISSION 1: LOGISTIC REGRESSION
# ==========================================
# Predict using the custom logistic regression model
lr_test_preds = model_lr.predict(X_test_scaled)

# Create the Kaggle-formatted DataFrame
submission_lr = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': lr_test_preds
})
# Export to CSV (index=False is strictly required by Kaggle)
submission_lr.to_csv('submission_logistic_regression.csv', index=False)
print("Logistic Regression submission saved!")

# ==========================================
# SUBMISSION 2: NEURAL NETWORK
# ==========================================
# Predict using the neural network (returns probabilities)
nn_test_probs = model_nn.predict(X_test_scaled).flatten()

# Convert probabilities to 0 or 1
nn_test_preds = (nn_test_probs > 0.5).astype(int)

# Create the Kaggle-formatted DataFrame
submission_nn = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': nn_test_preds
})
# Export to CSV
submission_nn.to_csv('submission_neural_network.csv', index=False)
print("Neural Network submission saved!")

Logistic Regression submission saved!
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Neural Network submission saved!


In [16]:
from sklearn.ensemble import RandomForestClassifier

# 1. Initialize the Random Forest model
# n_estimators=100 means it will build 100 separate decision trees
# max_depth=5 stops the trees from growing too deep, preventing overfitting
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

# 2. Train the model
# We will use the same SMOTE-balanced training data from your pipeline
rf_model.fit(X_train_balanced, y_train_balanced)

# 3. Predict on the test set
# (We can safely use X_test_scaled since we already processed it)
rf_test_preds = rf_model.predict(X_test_scaled)

# 4. Create the Kaggle-formatted DataFrame and export
submission_rf = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': rf_test_preds
})

submission_rf.to_csv('submission_random_forest.csv', index=False)
print("Random Forest submission saved!")

Random Forest submission saved!


In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import pandas as pd

# 1. Define a grid of settings to test
# The model will try every single combination of these numbers
param_grid = {
    'n_estimators': [50, 100, 200],       # Number of trees
    'max_depth': [5, 8, 12, None],        # How deep the trees can grow
    'min_samples_split': [2, 5, 10],      # Minimum samples required to split a node
    'min_samples_leaf': [1, 2, 4]         # Minimum samples required at a leaf node
}

# 2. Set up the Grid Search with 5-Fold Cross Validation (cv=5)
rf_base = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,                 # This is the K-Fold part (K=5)
    scoring='accuracy',   # Optimize for accuracy
    n_jobs=-1             # Use all your CPU cores to run this faster
)

# 3. Run the K-Fold Grid Search
# It will train 5 folds * 108 combinations = 540 models total to find the best one!
grid_search.fit(X_train_balanced, y_train_balanced)

print(f"Best parameters found by 5-Fold CV: {grid_search.best_params_}")
print(f"Best K-Fold accuracy score: {grid_search.best_score_:.4f}")

# 4. Predict on the test set using the guaranteed best model
best_rf_model = grid_search.best_estimator_
tuned_rf_preds = best_rf_model.predict(X_test_scaled)

# 5. Export for Kaggle
submission_tuned_rf = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': tuned_rf_preds
})
submission_tuned_rf.to_csv('submission_tuned_random_forest.csv', index=False)
print("Tuned Random Forest submission saved!")

Best parameters found by 5-Fold CV: {'max_depth': 12, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
Best K-Fold accuracy score: 0.8356
Tuned Random Forest submission saved!
